In [3]:
import pandas as pd

df = pd.read_csv("/Users/priyanka/Desktop/MLOPs/Mlops_Project_FinancialCrises/data/features/merged_features_clean_with_anomaly_flags_with_drift_flags.csv")

summary = {
    "rows": len(df),
    "cols": len(df.columns),
    "missing_ratio": df.isna().mean().mean(),
    "duplicate_rows": df.duplicated().sum(),
    "sector_balance": df['Sector'].value_counts(normalize=True).head(5).to_dict(),
}

summary


{'rows': 188670,
 'cols': 151,
 'missing_ratio': 0.0,
 'duplicate_rows': 0,
 'sector_balance': {'Technology': 0.2001643080510945,
  'Financials': 0.2001643080510945,
  'Consumer Staples': 0.1200985848306567,
  'Consumer Discretionary': 0.11927704457518419,
  'Industrials': 0.0800657232204378}}

In [4]:
import pandas as pd

df = pd.read_csv("/Users/priyanka/Desktop/MLOPs/Mlops_Project_FinancialCrises/data/features/merged_features_clean.csv")

summary = {
    "rows": len(df),
    "cols": len(df.columns),
    "missing_ratio": df.isna().mean().mean(),
    "duplicate_rows": df.duplicated().sum(),
    "sector_balance": df['Sector'].value_counts(normalize=True).head(5).to_dict(),
}

summary


{'rows': 188670,
 'cols': 133,
 'missing_ratio': 0.0,
 'duplicate_rows': 0,
 'sector_balance': {'Technology': 0.2001643080510945,
  'Financials': 0.2001643080510945,
  'Consumer Staples': 0.1200985848306567,
  'Consumer Discretionary': 0.11927704457518419,
  'Industrials': 0.0800657232204378}}

In [7]:
"""
Final Data Quality Validation Script
------------------------------------
Validates the merged dataset across:
1. Structural Consistency
2. Temporal Validity
3. Financial Logic
4. Statistical Sanity
5. Macroeconomic Realism
6. Feature Integrity
7. Cross-Sectional Consistency
8. Drift/Bias Risk (basic)
"""

import pandas as pd
import numpy as np
from scipy.stats import zscore, ks_2samp

# === LOAD DATA ===
df = pd.read_csv("/Users/priyanka/Desktop/MLOPs/Mlops_Project_FinancialCrises/data/features/merged_features_clean_with_anomaly_flags_with_drift_flags.csv")
print(f"\n📘 Loaded dataset with {len(df):,} rows and {len(df.columns)} columns\n")

results = []

def log(check_name, condition, details=""):
    symbol = "✅" if condition else "⚠️"
    results.append(f"{symbol} {check_name} {details}")

# === 1. STRUCTURAL CONSISTENCY ===
# === 1. STRUCTURAL CONSISTENCY ===
log("Unique (Date, Company) pairs", df.duplicated(subset=["Date", "Company"]).sum() == 0)
log("No duplicate rows", df.duplicated().sum() == 0)
log("No missing values", df.isna().sum().sum() == 0)

# Check that all non-ID columns are numeric
non_id_cols = [c for c in df.columns if c not in ["Date", "Company", "Sector"]]
numeric_cols = df.select_dtypes(include=["float", "int"]).columns.tolist()
non_numeric_cols = [c for c in non_id_cols if c not in numeric_cols]
log("All feature columns are numeric (except ID/Sector)", len(non_numeric_cols) == 0, f"{non_numeric_cols[:5]} ...")


# === 2. TEMPORAL VALIDITY ===
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])
    gaps = df.groupby("Company")["Date"].diff().dt.days
    long_gaps = (gaps > 30).sum()
    log("No long (>30d) date gaps", long_gaps == 0, f"({long_gaps} found)")
    monotonic = df.groupby("Company")["Date"].apply(lambda x: x.is_monotonic_increasing).all()
    log("Monotonic date order per company", monotonic)

# === 3. FINANCIAL ACCOUNTING LOGIC ===
eps = 1e-3
if {"Total_Assets","Total_Liabilities","Total_Equity"}.issubset(df.columns):
    assets_check = np.abs(df["Total_Assets"] - (df["Total_Liabilities"] + df["Total_Equity"])) / df["Total_Assets"]
    log("Assets ≈ Liabilities + Equity", (assets_check < 0.02).mean() > 0.98)

if {"Debt_to_Assets","Total_Debt","Total_Assets"}.issubset(df.columns):
    ratio_check = np.abs(df["Debt_to_Assets"] - (df["Total_Debt"]/df["Total_Assets"])) < eps
    log("Debt_to_Assets ratio correct", ratio_check.mean() > 0.95)

if {"Current_Ratio","Current_Assets","Current_Liabilities"}.issubset(df.columns):
    ratio_check = np.abs(df["Current_Ratio"] - (df["Current_Assets"]/df["Current_Liabilities"])) < eps
    log("Current_Ratio correct", ratio_check.mean() > 0.95)

if {"Profit_Margin","Net_Income","Revenue"}.issubset(df.columns):
    ratio_check = np.abs(df["Profit_Margin"] - (df["Net_Income"]/df["Revenue"])) < 0.01
    log("Profit_Margin correct", ratio_check.mean() > 0.95)

# === 4. STATISTICAL DISTRIBUTION CHECKS ===
num_cols = df.select_dtypes(include=[np.number])
high_corr = num_cols.corr().abs().stack().loc[lambda x: (x > 0.95) & (x < 1)].count() / 2
low_var = (num_cols.var() < 1e-6).sum()
outliers = ((np.abs(zscore(num_cols)) > 3).sum().sum()) / num_cols.size

log("Low feature correlation redundancy", high_corr < 100, f"({int(high_corr)} pairs >0.95 corr)")
log("No near-zero variance columns", low_var == 0, f"({low_var} found)")
log("Low outlier ratio", outliers < 0.02, f"({outliers*100:.2f}% outliers)")

# === 5. MACROECONOMIC REALISM ===
def check_range(col, low, high):
    if col in df.columns:
        invalid = ((df[col] < low) | (df[col] > high)).mean()
        log(f"{col} realistic range", invalid < 0.02, f"({invalid*100:.2f}% outside)")
        
macro_checks = {
    "GDP_Growth_90D": (-0.2, 0.2),
    "Inflation": (-0.02, 0.15),
    "Unemployment_Rate": (0, 0.25),
    "VIX": (0, 100),
    "Federal_Funds_Rate": (0, 0.15),
}
for col, (low, high) in macro_checks.items():
    check_range(col, low, high)

# === 6. FEATURE INTEGRITY CHECKS ===
# === 6. FEATURE INTEGRITY CHECKS ===
if {"Stock_Return_22D", "Close"}.issubset(df.columns):
    # compute 22-day returns per company using transform to preserve alignment
    df_sorted = df.sort_values(["Company", "Date"]).copy()
    returns_calc = df_sorted.groupby("Company")["Close"].transform(lambda x: (x / x.shift(22)) - 1)
    valid_ratio = (np.abs(df_sorted["Stock_Return_22D"] - returns_calc) < 0.05).mean()
    log("Stock_Return_22D consistent with Close", valid_ratio > 0.9, f"({valid_ratio*100:.1f}% within tolerance)")

if {"Stock_vs_MA50","Stock_Price","Stock_MA50"}.issubset(df.columns):
    calc_check = np.abs(df["Stock_vs_MA50"] - ((df["Stock_Price"] - df["Stock_MA50"])/df["Stock_MA50"])) < 0.05
    log("Stock_vs_MA50 formula correct", calc_check.mean() > 0.9)

# === 7. CROSS-SECTOR BALANCE ===
if "Sector" in df.columns:
    counts = df["Sector"].value_counts(normalize=True)
    dominant = (counts > 0.5).any()
    log("Sector distribution balanced", not dominant)

# === 8. SIMPLE DRIFT CHECK (SP500 vs historical) ===
if "SP500_Close" in df.columns:
    first_half = df["SP500_Close"].iloc[:len(df)//2]
    second_half = df["SP500_Close"].iloc[len(df)//2:]
    ks_stat, _ = ks_2samp(first_half, second_half)
    log("No extreme SP500 drift", ks_stat < 0.2, f"(KS={ks_stat:.2f})")

# === SUMMARY ===
print("\n🔍 VALIDATION SUMMARY:\n" + "\n".join(results))
passed = sum("✅" in r for r in results)
total = len(results)
print(f"\n✅ Passed {passed}/{total} checks ({(passed/total)*100:.1f}%)")

if passed/total > 0.85:
    print("\n🎯 Data quality looks solid — safe to proceed with model development.\n")
else:
    print("\n⚠️ Some issues found — review flagged checks before training.\n")



📘 Loaded dataset with 188,670 rows and 151 columns


🔍 VALIDATION SUMMARY:
✅ Unique (Date, Company) pairs 
✅ No duplicate rows 
✅ No missing values 
⚠️ All feature columns are numeric (except ID/Sector) ['Company_Name', 'VIX_Regime'] ...
✅ No long (>30d) date gaps (0 found)
✅ Monotonic date order per company 
⚠️ Assets ≈ Liabilities + Equity 
⚠️ Debt_to_Assets ratio correct 
✅ Current_Ratio correct 
⚠️ Profit_Margin correct 
⚠️ Low feature correlation redundancy (138 pairs >0.95 corr)
⚠️ No near-zero variance columns (1 found)
✅ Low outlier ratio (1.41% outliers)
⚠️ GDP_Growth_90D realistic range (89.23% outside)
⚠️ Inflation realistic range (42.79% outside)
⚠️ Unemployment_Rate realistic range (100.00% outside)
✅ VIX realistic range (0.00% outside)
⚠️ Federal_Funds_Rate realistic range (66.97% outside)
⚠️ Stock_Return_22D consistent with Close (0.7% within tolerance)
⚠️ Stock_vs_MA50 formula correct 
✅ Sector distribution balanced 
✅ No extreme SP500 drift (KS=0.04)

✅ Passed 10/22 c

In [ ]:
import numpy as np
df = pd.read_csv("/Users/priyanka/Desktop/MLOPs/Mlops_Project_FinancialCrises/data/raw/company_income_raw.csv")

checks = {
    "Debt_to_Equity": (0, 10),
    "Current_Ratio": (0, 5),
    "Profit_Margin": (-1, 1),
    "ROA": (-0.5, 0.5),
    "ROE": (-1, 1.5),
    "Stock_Return_22D": (-0.8, 0.8),
    "Stock_Volatility_22D": (0, 1),
    "Stock_vs_MA50": (-0.5, 0.5),
    "Stock_RSI_14D": (0, 100),
    "GDP_Growth_90D": (-0.1, 0.1),
    "Inflation": (-0.02, 0.15),
    "Unemployment_Rate": (0, 0.2),
    "Federal_Funds_Rate": (0, 0.1),
    "Yield_Curve_Spread": (-0.05, 0.05),
    "Oil_Price": (0, 200),
    "Consumer_Confidence": (30, 150),
    "VIX": (10, 80),
}

for col, (low, high) in checks.items():
    if col in df.columns:
        out = ((df[col] < low) | (df[col] > high)).mean()*100
        print(f"{col:25s} → {out:6.2f}% outside range")


Debt_to_Equity            →  18.13% outside range
Current_Ratio             →   6.47% outside range
Profit_Margin             →  97.87% outside range
ROA                       →  75.48% outside range
ROE                       →  92.70% outside range
Stock_Return_22D          →  88.44% outside range
Stock_Volatility_22D      → 100.00% outside range
Stock_vs_MA50             →  99.98% outside range
Stock_RSI_14D             →   0.00% outside range
GDP_Growth_90D            →  96.95% outside range
Inflation                 →  42.79% outside range
Unemployment_Rate         → 100.00% outside range
Federal_Funds_Rate        →  79.04% outside range
Yield_Curve_Spread        →  96.17% outside range
Oil_Price                 →   0.00% outside range
Consumer_Confidence       →   0.00% outside range
VIX                       →   1.30% outside range


In [18]:
df = pd.read_csv("/Users/priyanka/Desktop/MLOPs/Mlops_Project_FinancialCrises/data/raw/company_income_raw.csv")
df["Profit_Margin"] = df["Net_Income"] / df["Revenue"]
df["Profit_Margin"].head()

0    0.200181
1    0.214271
2    0.212504
3    0.209008
4    0.185138
Name: Profit_Margin, dtype: float64

In [20]:
df["Profit_Margin"].describe()

count    2016.000000
mean       -0.425243
std        23.849123
min     -1070.643836
25%         0.049475
50%         0.116909
75%         0.199808
max         1.219053
Name: Profit_Margin, dtype: float64

In [ ]:
df = pd.read_csv("/Users/priyanka/Desktop/MLOPs/Mlops_Project_FinancialCrises/data/features/merged_features_clean_with_anomaly_flags_with_drift_flags.csv")

In [27]:
df["Profit_Margin"].describe()

count    188670.000000
mean          5.433247
std          76.409542
min       -1000.000000
25%           4.895587
50%          11.544965
75%          19.782054
max         121.905297
Name: Profit_Margin, dtype: float64

In [24]:
df["Profit_Margin"].decsribe()

AttributeError: 'Series' object has no attribute 'decsribe'